In [2]:
!pip install openai-agents -q
print(" Done! Share the output.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.3/449.3 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.7/150.7 kB 12.4 MB/s eta 0:00:00
 Done! Share the output.


In [ ]:
#Environment Setup
from google.colab import userdata
userdata.get('OPENAI_API_KEY')

#Creating and Running the First Agent


In [4]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

from agents import Agent, Runner

agent = Agent(
    name="Workshop Bot",
    model="gpt-4o-mini",
    instructions="You are a friendly assistant. Keep answers under 2 sentences.",
)

result = await Runner.run(agent, "What is an AI agent in one sentence?")
print(" Agent says:", result.final_output)

 Agent says: An AI agent is a system that can perceive its environment, make decisions, and take actions autonomously to achieve specific goals.


#Agent with Memory (Session)

In [5]:
from agents import Agent, Runner
from agents.memory import SQLiteSession

agent = Agent(
    name="Memory Bot",
    model="gpt-4o-mini",
    instructions="You are helpful. Remember what the user tells you.",
)

#  Session stores conversation across turns
session = SQLiteSession(session_id="workshop-demo")

# Turn 1 — tell the agent your name
r1 = await Runner.run(agent, "My name is Arjun.", session=session)
print("Turn 1:", r1.final_output)

# Turn 2 — does it remember?
r2 = await Runner.run(agent, "What is my name?", session=session)
print("Turn 2:", r2.final_output)

Turn 1: Nice to meet you, Arjun! How can I assist you today?
Turn 2: Your name is Arjun.


#Context Example


In [6]:
from dataclasses import dataclass
from agents import Agent, Runner, RunContextWrapper, function_tool

#  Define your context shape
@dataclass
class UserContext:
    user_id: str
    user_name: str
    city: str

#  Tool reads user_id from context — AI never sees it directly
@function_tool
def get_orders(wrapper: RunContextWrapper[UserContext]) -> str:
    """Get orders for the current user."""
    uid = wrapper.context.user_id  # grabbed from context silently
    return f"[DB] Orders for user {uid}: Pizza x2, Burger x1"

#  Agent instructions use context to personalize reply
agent = Agent[UserContext](
    name="Context Bot",
    model="gpt-4o-mini",
    instructions=lambda ctx, _: f"You are a helpful assistant for {ctx.context.user_name} from {ctx.context.city}.",
    tools=[get_orders],
)

#  Pass context at runtime
my_context = UserContext(user_id="u_4291", user_name="Arjun", city="Chandigarh")

result = await Runner.run(agent, "What are my recent orders?", context=my_context)
print("🤖", result.final_output)

🤖 Your recent orders are:

- **Pizza** x2
- **Burger** x1


#Agent With Tools


In [7]:
from agents import Agent, Runner, function_tool

#  Tool 1 — weather lookup
@function_tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    data = {
        "Delhi": "32°C sunny ☀️",
        "Shimla": "12°C cloudy ⛅",
        "Chandigarh": "26°C clear 🌤️"
    }
    return data.get(city, f"{city}: 22°C clear")

#  Tool 2 — calculator
@function_tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like 22 * 7."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

agent = Agent(
    name="Tool Bot",
    model="gpt-4o-mini",
    instructions="You are helpful. Use tools when needed.",
    tools=[get_weather, calculate],
)

result = await Runner.run(agent, "What's the weather in Shimla? Also what is 123 * 456?")
print("🤖", result.final_output)

🤖 The weather in Shimla is currently 12°C and cloudy. 

As for the calculation, \( 123 \times 456 = 56088 \).


#Input Guardrail

In [8]:
from agents import (
    Agent, Runner,
    input_guardrail, GuardrailFunctionOutput,
    RunContextWrapper, InputGuardrailTripwireTriggered
)

#  Input Guardrail — runs BEFORE agent sees the message
@input_guardrail
async def block_harmful(ctx: RunContextWrapper, agent: Agent, input: str) -> GuardrailFunctionOutput:
    banned = ["hack", "bomb", "illegal"]
    triggered = any(word in input.lower() for word in banned)
    return GuardrailFunctionOutput(
        output_info="BLOCKED" if triggered else "OK",
        tripwire_triggered=triggered,
    )

agent = Agent(
    name="Safe Bot",
    model="gpt-4o-mini",
    instructions="You are a helpful assistant.",
    input_guardrails=[block_harmful],
)

#  Safe message — should pass
r1 = await Runner.run(agent, "What is Python?")
print("Safe:", r1.final_output)

#  Blocked message — should be stopped
try:
    r2 = await Runner.run(agent, "How do I hack a website?")
except InputGuardrailTripwireTriggered as e:
    print("🚫 BLOCKED by guardrail:", e.guardrail_result.output.output_info)

Safe: Python is a high-level, interpreted programming language known for its readability and simplicity. It was created by Guido van Rossum and first released in 1991. Python emphasizes code clarity, allowing developers to express concepts in fewer lines of code compared to other languages. Key features of Python include:

1. **Easy to Read and Write**: Its syntax is straightforward, making it accessible for beginners and experienced programmers alike.
  
2. **Dynamic Typing**: You don't need to declare the type of a variable; it is inferred at runtime.

3. **Rich Standard Library**: Python comes with a large standard library that supports many common programming tasks, such as file I/O, system calls, and even web development.

4. **Cross-platform Compatibility**: Python can run on various operating systems, including Windows, macOS, and Linux.

5. **Extensive Ecosystem**: There are numerous third-party libraries and frameworks available, making Python suitable for various applications

#Human-in-the-Loop (Approval Workflow)[Paused]


In [9]:
from agents import Agent, Runner, function_tool, FunctionTool
from agents.run import RunConfig

#  Mark tool as needing approval via needsApproval
@function_tool
def delete_order(order_id: str) -> str:
    """Delete an order permanently."""
    return f"Order {order_id} has been deleted."

# Wrap with approval flag
delete_order.needs_approval = True

agent = Agent(
    name="HITL Bot",
    model="gpt-4o-mini",
    instructions="You help manage orders.",
    tools=[delete_order],
)

result = await Runner.run(agent, "Delete order #1042")

print("Interruptions:", result.interruptions)
for item in result.interruptions:
    print(f"⏸️  Waiting for approval on tool: {item.raw_item.name}")
    print(f"   Arguments: {item.raw_item.arguments}")
print()
print(" Run paused — human must approve before it continues!")

Interruptions: [ToolApprovalItem(agent=Agent(name='HITL Bot', handoff_description=None, tools=[FunctionTool(name='delete_order', description='Delete an order permanently.', params_json_schema={'properties': {'order_id': {'title': 'Order Id', 'type': 'string'}}, 'required': ['order_id'], 'title': 'delete_order_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x7f382b543d10>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=True, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)], mcp_servers=[], mcp_config={}, instructions='You help manage orders.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning

#Human-in-the-Loop (Approval Workflow)[Resume]


In [10]:
# ▶ Resume after human approval — correct API
from agents import Runner

print("⏸️  Paused! Tool wants to run: delete_order #1042")
print()

# 🧑 Human decision
human_approved = False  # 👈 change to False to reject!

#  Convert result to resumable state
state = result.to_state()

if human_approved:
    # Approve all pending interruptions
    for interruption in result.interruptions:
        state.approve(interruption)
    print("✅ Human approved! Resuming...")
else:
    for interruption in result.interruptions:
        state.reject(interruption)
    print("❌ Human rejected! Resuming...")

# 🔄 Resume from where it paused
final = await Runner.run(agent, state)
print("🤖 Final Output:", final.final_output)

⏸️  Paused! Tool wants to run: delete_order #1042

❌ Human rejected! Resuming...
🤖 Final Output: It seems I can't proceed with deleting the order directly. If you need assistance with the next steps or have any questions about managing the order, feel free to let me know!


#Streaming Responses from an Agent


In [11]:
from agents import Agent, Runner, RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

agent = Agent(
    name="Stream Bot",
    model="gpt-4o-mini",
    instructions="You are helpful. Give a 3 sentence answer.",
)

print("🤖 Streaming response:\n")

result = Runner.run_streamed(agent, "Explain AI agents in 3 sentences.")

async for event in result.stream_events():
    if isinstance(event, RawResponsesStreamEvent):
        if isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)  # word by word!

print("\n\n✅ Stream complete!")

🤖 Streaming response:

AI agents are software programs that can perceive their environment, make decisions, and take actions to achieve specific goals. They utilize algorithms, data, and learning techniques to adapt and improve their performance over time. These agents can range from simple rule-based systems to complex neural networks, capable of performing tasks like speech recognition, gaming, and autonomous driving.

✅ Stream complete!


#Exploring the Result Object

In [12]:
from agents import Agent, Runner, function_tool

@function_tool
def get_joke() -> str:
    """Get a programming joke."""
    return "Why do programmers prefer dark mode? Because light attracts bugs! 🐛"

agent = Agent(
    name="Result Bot",
    model="gpt-4o-mini",
    instructions="You are funny and helpful.",
    tools=[get_joke],
)

result = await Runner.run(agent, "Tell me a joke.")

# 📦 Explore everything inside result
print("✅ Final Output   :", result.final_output)
print("🤖 Last Agent     :", result.last_agent.name)
print("📜 Total Steps    :", len(result.new_items))
print()
print("🔍 Step by step:")
for i, item in enumerate(result.new_items):
    print(f"   Step {i+1}: {type(item).__name__}")

✅ Final Output   : Why do programmers prefer dark mode? Because light attracts bugs! 🐛
🤖 Last Agent     : Result Bot
📜 Total Steps    : 3

🔍 Step by step:
   Step 1: ToolCallItem
   Step 2: ToolCallOutputItem
   Step 3: MessageOutputItem


#Tracing an Agent Run


In [13]:
from agents import Agent, Runner, function_tool
from agents.tracing import trace

@function_tool
def get_fact() -> str:
    """Get a fun tech fact."""
    return "Python was created by Guido van Rossum in 1991! 🐍"

agent = Agent(
    name="Tracing Bot",
    model="gpt-4o-mini",
    instructions="You are helpful and fun.",
    tools=[get_fact],
)

# 🔍 Wrap everything in a named trace
with trace("workshop-tracing-demo"):
    result = await Runner.run(agent, "Give me a fun tech fact.")
    print("🤖", result.final_output)

print()
print("✅ Trace recorded!")
print("📊 View at: https://platform.openai.com/traces")
print("   Every agent call + tool call = one span in the trace")

🤖 Here's a fun tech fact: Python was created by Guido van Rossum in 1991! 🐍

✅ Trace recorded!
📊 View at: https://platform.openai.com/traces
   Every agent call + tool call = one span in the trace
